# TrustLens — Experimental Case Selection
### Selecting and Documenting the Six Study Cases

---

**Project:** *TrustLens — Trust Calibration, Decision Agency, and Fairness Perception Across Explanation Modalities in AI-Assisted Loan Decision-Making*

This notebook selects the six loan cases that form the experimental stimulus of the user study, and exports them as a cleanly formatted spreadsheet suitable for documentation and version control.

The six cases are chosen to satisfy two layered requirements simultaneously. First, a structural requirement: the cases must span the full two-by-two of AI correctness (correct / incorrect) and AI confidence (high / low), so that the trust-calibration and override analyses observe every relevant situation. Second, a fairness requirement: a subset of the cases must surface a demographic-proxy feature (age or personal status / sex) among their leading SHAP drivers, providing the stimulus required to study fairness perception.

Because the fairness requirement depends on each case's SHAP attribution, case selection is performed only after the explanation pipeline is in place.

## 1. Environment and Artefacts

In [1]:
!pip install xgboost shap xlsxwriter -q

import pandas as pd
import numpy as np
import joblib
import shap
import warnings
warnings.filterwarnings('ignore')

from google.colab import files
print('Upload: xgboost_model.pkl, decision_threshold.pkl, X_test.csv, y_test.csv,')
print('and german_credit_processed.csv')
uploaded = files.upload()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 15.7 MB/s eta 0:00:00
Upload: xgboost_model.pkl, decision_threshold.pkl, X_test.csv, y_test.csv,
and german_credit_processed.csv


Saving X_test.csv to X_test.csv
Saving xgboost_model.pkl to xgboost_model.pkl
Saving y_test.csv to y_test.csv
Saving decision_threshold.pkl to decision_threshold.pkl


In [2]:
model     = joblib.load('xgboost_model.pkl')
threshold = joblib.load('decision_threshold.pkl')
X_test    = pd.read_csv('X_test.csv')
y_test    = pd.read_csv('y_test.csv').squeeze()
print('Loaded. Test set:', X_test.shape, '| threshold:', round(float(threshold),3))

Loaded. Test set: (200, 20) | threshold: 0.51


## 2. Reconstruct Readable Feature Values

The model operates on encoded data, but the study cases must be displayed to participants in human-readable form. We reconstruct the decoded German Credit values from the original source so that each selected case carries its full, readable applicant profile.

*Upload the original dataset file when prompted.*

In [3]:
print('Upload the original dataset: german_credit_with_codebook__TrustLens_.xlsx')
uploaded2 = files.upload()
raw_file = [k for k in uploaded2.keys() if k.endswith('.xlsx')][0]

df_raw = pd.read_excel(raw_file, sheet_name='German Credit Data')
df_codebook = pd.read_excel(raw_file, sheet_name='Codebook')

column_name_fix = {'savings_account':'savings_account_bonds','other_debtors':'other_debtors_guarantors'}
valid_cols = set(df_raw.columns) - {'credit_risk'}
decode_map = {}
for _, row in df_codebook.iterrows():
    col=str(row['Column']).strip(); code=str(row['Code']).strip(); label=str(row['Meaning / Readable Label']).strip()
    col=column_name_fix.get(col,col)
    if col in valid_cols and code not in ['nan',''] and label not in ['nan','']:
        decode_map.setdefault(col,{})[code]=label

df_readable = df_raw.copy()
for col,mapping in decode_map.items():
    df_readable[col]=df_readable[col].astype(str).map(mapping).fillna(df_readable[col].astype(str))
print('Readable profiles reconstructed.')

Upload the original dataset: german_credit_with_codebook__TrustLens_.xlsx


Saving german_credit_with_codebook__TrustLens_.xlsx to german_credit_with_codebook__TrustLens_.xlsx
Readable profiles reconstructed.


## 3. Categorise the Test Set

For every test applicant we record the AI's decision, its confidence, whether it was correct, and whether a demographic-proxy feature appears among its top three SHAP drivers. These properties are what the two selection requirements operate on.

In [4]:
explainer   = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

fairness_feats = ['personal_status_sex', 'age_years']
feat_names = list(X_test.columns)
proba = model.predict_proba(X_test)[:, 1]
pred  = (proba >= threshold).astype(int)

pool = []
for i in range(len(X_test)):
    actual_bad = (y_test.values[i] == 1)
    decision = 'REJECT' if pred[i]==1 else 'APPROVE'
    conf = proba[i] if pred[i]==1 else 1-proba[i]
    correct = (pred[i]==1)==actual_bad
    top3 = [feat_names[j] for j in np.argsort(np.abs(shap_values[i]))[::-1][:3]]
    pool.append({'position':i,'decision':decision,'confidence':conf,
                 'correct':correct,'fairness':any(f in top3 for f in fairness_feats)})
pool = pd.DataFrame(pool)
print('Categorised', len(pool), 'cases.')

Categorised 200 cases.


## 4. The Selected Six Cases

The following six test positions were selected. Each is annotated with the research role it serves. Together they fill all four correctness-by-confidence cells, and four of the six carry a demographic-proxy driver.

In [5]:
selected = {
    100: ('1', 'Correct-Confident', 'Clean baseline'),
    45 : ('2', 'Correct-Confident', 'Baseline with age as a driver'),
    22 : ('3', 'Wrong-Confident',   'Overtrust trap, no demographic driver'),
    94 : ('4', 'Wrong-Confident',   'Overtrust trap with sex as a driver'),
    58 : ('5', 'Correct-Uncertain', 'Undertrust test with sex as a driver'),
    124: ('6', 'Wrong-Uncertain',   'Wrong rejection with age as a driver'),
}

for pos,(num,ctype,role) in selected.items():
    r = pool[pool['position']==pos].iloc[0]
    print(f"Case {num} [{ctype}] pos={pos}: {r['decision']} "
          f"{int(round(r['confidence']*100))}% | correct={r['correct']} | fairness={r['fairness']}")

Case 1 [Correct-Confident] pos=100: APPROVE 98% | correct=True | fairness=False
Case 2 [Correct-Confident] pos=45: APPROVE 97% | correct=True | fairness=True
Case 3 [Wrong-Confident] pos=22: APPROVE 96% | correct=False | fairness=False
Case 4 [Wrong-Confident] pos=94: APPROVE 81% | correct=False | fairness=True
Case 5 [Correct-Uncertain] pos=58: REJECT 69% | correct=True | fairness=True
Case 6 [Wrong-Uncertain] pos=124: REJECT 53% | correct=False | fairness=True


## 5. Verify the Design

Two checks confirm validity: the correctness pattern must match the one used to validate the calibration metric, and at least four of six cases must be fairness-loaded.

In [6]:
ordered = sorted(selected.items(), key=lambda kv: kv[1][0])
pattern = [int(pool[pool['position']==pos].iloc[0]['correct']) for pos,_ in ordered]
fairness_count = sum(int(pool[pool['position']==pos].iloc[0]['fairness']) for pos,_ in ordered)
print('Correctness pattern :', pattern, '(expected [1, 1, 0, 0, 1, 0])')
print('Match               :', pattern == [1,1,0,0,1,0])
print('Fairness-loaded cases:', fairness_count, 'of 6')

Correctness pattern : [1, 1, 0, 0, 1, 0] (expected [1, 1, 0, 0, 1, 0])
Match               : True
Fairness-loaded cases: 4 of 6


## 6. Assemble the Clean Study-Case Table

We build a presentation-quality table: human-friendly column names, a logical column order (research metadata first, then the most decision-relevant applicant features), and readable values throughout.

In [7]:
display_names = {
    'case_number':'Case #', 'case_type':'Case Type', 'case_role':'Research Role',
    'ai_decision':'AI Decision', 'ai_confidence':'AI Confidence',
    'actual_outcome':'Actual Outcome', 'ai_correct':'AI Correct?',
    'checking_account_status':'Checking Account', 'credit_amount':'Credit Amount (DM)',
    'duration_months':'Loan Duration (months)', 'credit_history':'Credit History',
    'purpose':'Loan Purpose', 'savings_account_bonds':'Savings Account',
    'employment_since':'Employment Duration', 'age_years':'Age',
    'personal_status_sex':'Personal Status / Sex', 'property':'Property',
    'housing':'Housing', 'job':'Job',
    'installment_rate_percent_disposable_income':'Installment Rate (%)',
    'other_debtors_guarantors':'Other Debtors', 'residence_since_years':'Residence (years)',
    'other_installment_plans':'Other Installment Plans',
    'existing_credits_at_bank':'Existing Credits',
    'people_liable_for_maintenance':'Dependents', 'telephone':'Telephone',
    'foreign_worker':'Foreign Worker'
}
ordered_cols = ['case_number','case_type','case_role','ai_decision','ai_confidence',
    'actual_outcome','ai_correct','checking_account_status','credit_amount','duration_months',
    'credit_history','purpose','savings_account_bonds','employment_since','age_years',
    'personal_status_sex','property','housing','job','installment_rate_percent_disposable_income',
    'other_debtors_guarantors','residence_since_years','other_installment_plans',
    'existing_credits_at_bank','people_liable_for_maintenance','telephone','foreign_worker']

rows = []
for pos,(num,ctype,role) in selected.items():
    oi = X_test.index[pos]
    pb = model.predict_proba(X_test.iloc[[pos]])[0,1]
    dec = 'REJECT' if pb>=threshold else 'APPROVE'
    conf = pb if dec=='REJECT' else 1-pb
    actual_bad = y_test.values[pos]==1
    correct = (dec=='REJECT')==actual_bad
    r = df_readable.loc[oi].to_dict()
    r.update({'case_number':num,'case_type':ctype,'case_role':role,'ai_decision':dec,
              'ai_confidence':f'{int(round(conf*100))}%',
              'actual_outcome':'Bad' if actual_bad else 'Good',
              'ai_correct':'Yes' if correct else 'No'})
    rows.append(r)

study = pd.DataFrame(rows)[ordered_cols].rename(columns=display_names)
study = study.sort_values('Case #').reset_index(drop=True)
study

,Case #,Case Type,Research Role,AI Decision,AI Confidence,Actual Outcome,AI Correct?,Checking Account,Credit Amount (DM),Loan Duration (months),...,Housing,Job,Installment Rate (%),Other Debtors,Residence (years),Other Installment Plans,Existing Credits,Dependents,Telephone,Foreign Worker
0,1,Correct-Confident,Clean baseline,APPROVE,98%,Good,Yes,no checking account,1469,24,...,rent,unskilled resident,4,none,4,none,1,1,none,yes
1,2,Correct-Confident,Baseline with age as a driver,APPROVE,97%,Good,Yes,no checking account,1393,11,...,own,management / self-employed / highly qualified ...,4,none,4,none,2,1,none,yes
2,3,Wrong-Confident,"Overtrust trap, no demographic driver",APPROVE,96%,Bad,No,< 0 DM,2241,10,...,rent,unskilled resident,1,none,3,none,2,2,none,no
3,4,Wrong-Confident,Overtrust trap with sex as a driver,APPROVE,81%,Bad,No,0 <= balance < 200 DM,1318,12,...,own,skilled employee / official,4,none,4,none,1,1,"yes, registered under customer name",yes
4,5,Correct-Uncertain,Undertrust test with sex as a driver,REJECT,69%,Bad,Yes,>= 200 DM / salary assignment for at least 1 year,1961,18,...,own,management / self-employed / highly qualified ...,3,none,2,none,1,1,none,yes
5,6,Wrong-Uncertain,Wrong rejection with age as a driver,REJECT,53%,Good,No,0 <= balance < 200 DM,1924,18,...,rent,skilled employee / official,4,none,3,none,1,1,none,yes


## 7. Export — Formatted Excel and Clean CSV

We export two artefacts. The Excel file is presentation-quality: a styled header, auto-sized and wrapped columns, frozen identifier columns, and no cramped cells. The CSV is a clean, properly-quoted machine-readable copy for the application to load.

In [8]:
with pd.ExcelWriter('study_cases.xlsx', engine='xlsxwriter') as writer:
    study.to_excel(writer, sheet_name='Study Cases', index=False, startrow=1)
    wb = writer.book; ws = writer.sheets['Study Cases']
    title_fmt  = wb.add_format({'bold':True,'font_size':14,'font_color':'#1a2e4a'})
    header_fmt = wb.add_format({'bold':True,'font_color':'white','bg_color':'#1a2e4a',
                                'align':'center','valign':'vcenter','text_wrap':True,'border':1})
    text_fmt   = wb.add_format({'align':'left','valign':'vcenter','text_wrap':True,'border':1})
    center_fmt = wb.add_format({'align':'center','valign':'vcenter','border':1})

    ws.write(0,0,'TrustLens — Six Experimental Study Cases', title_fmt)
    for c, value in enumerate(study.columns):
        ws.write(1, c, value, header_fmt)
    for i, col in enumerate(study.columns):
        width = min(max(study[col].astype(str).map(len).max(), len(col)) + 3, 28)
        ws.set_column(i, i, width)
    for r in range(len(study)):
        for c in range(len(study.columns)):
            fmt = center_fmt if c < 7 else text_fmt
            ws.write(r+2, c, study.iloc[r, c], fmt)
    ws.set_row(1, 30)
    ws.freeze_panes(2, 3)

study.to_csv('study_cases.csv', index=False)
print('Exported study_cases.xlsx (formatted) and study_cases.csv (clean)')
files.download('study_cases.xlsx')
files.download('study_cases.csv')

Exported study_cases.xlsx (formatted) and study_cases.csv (clean)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## Summary

Six experimental cases were selected through a layered, documented procedure. The structural requirement guarantees the set spans every combination of AI correctness and confidence, supporting the trust-calibration and override analyses. The fairness requirement guarantees that four of the six cases surface a demographic-proxy feature, supporting the fairness-perception analysis. The correctness pattern matches the one against which the calibration metric was validated, aligning the experimental stimulus with the measurement instrument.

**Artefacts**
- `study_cases.xlsx` — presentation-quality, for documentation and review
- `study_cases.csv` — clean machine-readable copy, for the application

Both belong in `study/cases/`.

**Version control**
```bash
git add . && git commit -m "Experimental case selection for trust calibration study" && git push
```